In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual
sns.set_theme(style="whitegrid")

# 1. Carga de Datos
url = "https://raw.githubusercontent.com/readytensor/rt-datasets-binary-classification/refs/heads/main/datasets/processed/credit_approval/credit_approval.csv"

column_names = [
    "gender", "age", "debt_ratio", "marital_status", "customer_type",
    "occupation", "employment_status", "years_employed", "has_default_history",
    "owns_assets", "credit_score", "has_other_loans", "citizenship",
    "annual_income", "balance", "approved"
]

df = pd.read_csv(url, names=column_names, header=0)

# Mostramos las primeras filas para entender la estructura
df.sample(10)

### Pre & Filtering

In [ ]:
# 2.1. Corrección Manual de Valores Ilegales
# Asumimos que no puede haber edades negativas o ingresos negativos.
df = df[(df['age'] >= 0) & (df['annual_income'] >= 0)]

# 2.2. Tratamiento de Outliers (Capping / Winsorization)
# Limitamos los ingresos atípicos al percentil 99 para no distorsionar el modelo
percentil_99 = df['annual_income'].quantile(0.99)
df['annual_income'] = np.where(df['annual_income'] > percentil_99, percentil_99, df['annual_income'])


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Separar características numéricas y categóricas
numeric_features = ['age', 'debt_ratio', 'years_employed', 'credit_score', 'annual_income', 'balance']
categorical_features = ['gender', 'marital_status', 'customer_type', 'occupation', 'employment_status', 'has_default_history', 'owns_assets', 'has_other_loans', 'citizenship']

# Pipeline para variables numéricas: Imputación de faltantes por la mediana y escalado
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Filtro: Imputación 
    ('scaler', StandardScaler()) # Variables Numéricas StandardScaler 
])

# Pipeline para variables categóricas: Imputación por la moda y codificación binaria
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')) # One-Hot Encoding 
])

# Ensamblar el preprocesador
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])


In [ ]:
preprocessor

### Clasificacion

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

# 1. División de Datos (Train/Test)
X_clf = df.drop('approved', axis=1)
y_clf = np.where(df['approved'] == 'positive', 1, 0)

# Train (80%) / Test (20%) 
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42)

# 2. Definición de Modelos
modelos_clasificacion = {
    "Regresión Logística": LogisticRegression(max_iter=1000), # Rápido, interpretable 
    "Árbol de Decisión": DecisionTreeClassifier(random_state=42), # Intuitivo y visual 
    "Random Forest": RandomForestClassifier(random_state=42) # Robusto ante outliers 
}

# Configuración del lienzo para las Matrices de Confusión (1 fila, 3 columnas)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Diagnóstico Visual: Matrices de Confusión por Modelo', fontsize=16)

# Configuración del lienzo para la Curva ROC comparativa
fig_roc, roc_ax = plt.subplots(figsize=(10, 8))
roc_ax.set_title('Comparativa de Curvas ROC', fontsize=16)
roc_ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Azar (AUC = 0.50)')

# 3. Entrenamiento, Evaluación y Visualización
for idx, (nombre, modelo) in enumerate(modelos_clasificacion.items()):
    # Estructurar Pipeline
    clf_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                   ('classifier', modelo)])
    
    # Entrenar
    clf_pipeline.fit(X_train_clf, y_train_clf)
    
    # Predecir clases y probabilidades
    y_pred_clf = clf_pipeline.predict(X_test_clf)
    y_pred_proba = clf_pipeline.predict_proba(X_test_clf)[:, 1]
    
    # --- Reporte en Consola ---
    print(f"--- {nombre} ---")
    print(f"Reporte de Clasificación:\n{classification_report(y_test_clf, y_pred_clf)}")
    auc_score = roc_auc_score(y_test_clf, y_pred_proba)
    print(f"AUC ROC: {auc_score:.4f}\n")
    
    # --- Gráfico 1: Matriz de Confusión (Heatmap) ---
    cm = confusion_matrix(y_test_clf, y_pred_clf)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False,
                annot_kws={"size": 14})
    axes[idx].set_title(f'{nombre}')
    axes[idx].set_xlabel('Predicción del Modelo')
    axes[idx].set_ylabel('Valor Real')
    
    # --- Gráfico 2: Curva ROC ---
    fpr, tpr, umbrales = roc_curve(y_test_clf, y_pred_proba)
    roc_ax.plot(fpr, tpr, linewidth=2, label=f'{nombre} (AUC = {auc_score:.4f})')

# Ajustes finales y renderizado de las gráficas
plt.tight_layout()

# Mostrar Matrices de Confusión
plt.show()

# Ajustes visuales para la Curva ROC
roc_ax.set_xlabel('Tasa de Falsos Positivos (FPR)', fontsize=12)
roc_ax.set_ylabel('Tasa de Verdaderos Positivos (TPR)', fontsize=12)
roc_ax.legend(loc='lower right', fontsize=12)
roc_ax.grid(True, alpha=0.3)

# Mostrar Curva ROC comparativa
plt.figure(fig_roc.number)
plt.show()

### Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Modificar el Dataset para Regresión
# Target = 'credit_score'. Eliminamos 'approved' para evitar correlaciones espurias o sesgos.
X_reg = df.drop(['credit_score', 'approved'], axis=1)
y_reg = df['credit_score']

# Actualizamos las listas de características numéricas para el preprocesador
numeric_features_reg = ['age', 'debt_ratio', 'years_employed', 'annual_income', 'balance']

preprocessor_reg = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features_reg),
        ('cat', categorical_transformer, categorical_features)
    ])

# 2. División Train/Test
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# 3. Pipeline de Regresión Lineal
reg_pipeline = Pipeline(steps=[('preprocessor', preprocessor_reg),
                               ('regressor', LinearRegression())]) # El modelo base 

# 4. Entrenamiento y Predicción
reg_pipeline.fit(X_train_reg, y_train_reg)
y_pred_reg = reg_pipeline.predict(X_test_reg)

# 5. Panel de Evaluación: Regresión 
mae = mean_absolute_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
r2 = r2_score(y_test_reg, y_pred_reg)

print("--- Evaluación Regresión Lineal ---")
print(f"MAE: {mae:.2f}") # Error promedio en unidades reales 
print(f"RMSE: {rmse:.2f}") # Penaliza fuertemente los errores grandes 
print(f"R²: {r2:.4f}") # Proporción de variabilidad explicada 


In [ ]:
# Calcular los residuos
residuos = y_test_reg - y_pred_reg

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de Residuos
sns.scatterplot(x=y_pred_reg, y=residuos, ax=ax[0], alpha=0.6)
ax[0].axhline(y=0, color='r', linestyle='--')
ax[0].set_title('Gráfico de Residuos')
ax[0].set_xlabel('Valores Predichos')
ax[0].set_ylabel('Residuos')
# Interpretación: Si hay un patrón (ej. embudo), el modelo falla.

# Histograma de Residuos
sns.histplot(residuos, kde=True, ax=ax[1], bins=30)
ax[1].set_title('Histograma de Residuos')
ax[1].set_xlabel('Residuos')
# Interpretación: Los errores deben distribuirse simétricamente (distribución normal).

plt.tight_layout()
plt.show()



### Cross-Validation Classification

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

# Definimos el esquema de validación cruzada (5 folds)
cv_estrategia_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("--- Validación Cruzada: Clasificación (Métrica AUC ROC) ---")

for nombre, modelo in modelos_clasificacion.items():
    # Creamos el pipeline completo para cada modelo
    pipeline_eval = Pipeline(steps=[('preprocessor', preprocessor),
                                    ('classifier', modelo)])
    
    # cross_val_score ejecuta internamente el ajuste (fit) del preprocesamiento 
    # y del modelo de forma aislada en cada fold, previniendo el Data Leakage.
    scores_cv = cross_val_score(pipeline_eval, X_clf, y_clf, cv=cv_estrategia_clf, scoring='roc_auc')
    
    print(f"{nombre}:")
    print(f"  Scores por fold: {scores_cv}")
    print(f"  AUC Promedio: {scores_cv.mean():.4f} (+/- desviación estándar: {scores_cv.std() * 2:.4f})\n")

### Cross-Validation Regression

In [ ]:
from sklearn.model_selection import KFold, cross_validate

# Definimos el esquema de validación cruzada para regresión
cv_estrategia_reg = KFold(n_splits=5, shuffle=True, random_state=42)

# Usamos cross_validate para evaluar múltiples métricas simultáneamente
metricas = ['neg_mean_absolute_error', 'r2']
resultados_cv = cross_validate(reg_pipeline, X_reg, y_reg, cv=cv_estrategia_reg, scoring=metricas)

# scikit-learn devuelve los errores en formato negativo para maximizarlos internamente, los convertimos a positivo
mae_scores = -resultados_cv['test_neg_mean_absolute_error']
r2_scores = resultados_cv['test_r2']

print("--- Validación Cruzada: Regresión Lineal ---")
print(f"MAE Promedio: {mae_scores.mean():.2f} (+/- {mae_scores.std() * 2:.2f})")
print(f"R² Promedio: {r2_scores.mean():.4f} (+/- {r2_scores.std() * 2:.4f})")


### [EXTRA] Pues por que nadie te lo explica en la primera instancia
> Andru dijo esto jeje -> yo

In [ ]:
import joblib
import json
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline

# 1. Definir la estrategia de validación cruzada
cv_estrategia_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Variables para rastrear el mejor modelo y sus métricas
mejor_score = 0.0
mejor_nombre = ""
mejor_pipeline = None
metricas_finales = {}

print("--- Evaluando y seleccionando el mejor modelo ---")

for nombre, modelo in modelos_clasificacion.items():
    # Estructurar el Pipeline con el preprocesador que definimos antes
    pipeline_actual = Pipeline(steps=[('preprocessor', preprocessor),
                                      ('classifier', modelo)])
    
    # Evaluar con validación cruzada
    scores_cv = cross_val_score(pipeline_actual, X_clf, y_clf, cv=cv_estrategia_clf, scoring='roc_auc')
    auc_promedio = scores_cv.mean()
    desviacion_estandar = scores_cv.std()
    
    # Entrenar el pipeline con TODOS los datos para el despliegue final
    pipeline_actual.fit(X_clf, y_clf)
    
    # Almacenar las métricas de este modelo en un diccionario
    metricas_finales[nombre] = {
        "auc_roc_promedio": round(float(auc_promedio), 4),
        "desviacion_estandar": round(float(desviacion_estandar), 4),
        "scores_por_fold": [round(float(s), 4) for s in scores_cv]
    }
    
    # Lógica de selección: si es el mejor hasta ahora, lo guardamos
    if auc_promedio > mejor_score:
        mejor_score = auc_promedio
        mejor_nombre = nombre
        mejor_pipeline = pipeline_actual

print(f"\nEl mejor modelo es: {mejor_nombre} con un AUC ROC promedio de {mejor_score:.4f}")

# 2. Guardar el objeto del Pipeline completo (.joblib)
# Esto guarda el preprocesamiento (escaladores, imputadores, codificadores) y el modelo en un solo archivo.
archivo_modelo = "mejor_modelo_credito.joblib"
joblib.dump(mejor_pipeline, archivo_modelo)
print(f"Pipeline del mejor modelo guardado con éxito en: '{archivo_modelo}'")

# 3. Guardar el diccionario de métricas de todos los modelos (.json)
archivo_metricas = "metricas_modelos.json"
metadata = {
    "mejor_modelo_seleccionado": mejor_nombre,
    "criterio_seleccion": "Maximo AUC ROC en Validacion Cruzada de 5 folds",
    "metricas_detalladas": metricas_finales
}

with open(archivo_metricas, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)
print(f"Métricas e historial de experimentos guardados con éxito en: '{archivo_metricas}'")

### Usar el modelo guardado

In [ ]:
import joblib
import json
import pandas as pd

# 1. Cargar las métricas para auditoría o reportes
with open("metricas_modelos.json", "r", encoding="utf-8") as f:
    metadata_cargada = json.load(f)

print("--- Información del Modelo Cargado ---")
print(f"Modelo en uso: {metadata_cargada['mejor_modelo_seleccionado']}")
print(f"Desempeño esperado (AUC): {metadata_cargada['metricas_detalladas'][metadata_cargada['mejor_modelo_seleccionado']]['auc_roc_promedio']}")

# 2. Cargar el Pipeline del modelo
modelo_produccion = joblib.load("mejor_modelo_credito.joblib")

# 3. Hacer predicciones directamente con datos completamente nuevos (crudos)
# El pipeline se encarga de imputar, codificar y escalar de manera automática y segura
nuevos_clientes = pd.DataFrame([{
    "gender": "b",
    "age": 32.5,
    "debt_ratio": 4.75,
    "marital_status": "u",
    "customer_type": "g",
    "occupation": "c",
    "employment_status": "v",
    "years_employed": 1.5,
    "has_default_history": "t",
    "owns_assets": "g",
    "credit_score": 2,
    "has_other_loans": "f",
    "citizenship": "g",
    "annual_income": 45000,
    "balance": 1200
}])

# Obtener predicción directa (Aprobado = 1, Rechazado = 0)
prediccion = modelo_produccion.predict(nuevos_clientes)
# Obtener probabilidades
probabilidades = modelo_produccion.predict_proba(nuevos_clientes)

print(f"\nPredicción para el nuevo cliente: {prediccion[0]}")
print(f"Probabilidad de aprobación: {probabilidades[0][1]:.4f}")